In [37]:
import re
import os
import base64
import fitz  
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pdf2markdown4llm import PDF2Markdown4LLM

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [38]:
input_pdf = "../data/Machine_Learning_Wikipedia.pdf"
input_MD = "../output.md"
output_MD = "../output_cleaned_final.md"
image_DIR = "../images"

os.makedirs(image_DIR, exist_ok=True)

In [39]:
converter = PDF2Markdown4LLM(remove_headers=False)
markdown = converter.convert(input_pdf)

with open(input_MD, "w", encoding="utf-8") as f:
    f.write(markdown)

print(" PDF converted to Markdown")


Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


 PDF converted to Markdown


In [40]:
def images_to_files(input_pdf):
    doc = fitz.open(input_pdf)
    extracted = []
    idx = 0 

    for page_num, page in enumerate(doc):
        for img in page.get_images(full=True):
            xref = img[0]
            data = doc.extract_image(xref) 
            ext = data["ext"]
            img_bytes = data["image"]

            file_name = f"img_{idx:03d}.{ext}"
            file_path = os.path.join(image_DIR, file_name)

            with open(file_path, "wb") as f:
                f.write(img_bytes)

            extracted.append({"page": page_num + 1, "file": file_path})
            idx += 1

    return extracted

images = images_to_files(input_pdf)

print(f"Extracted {len(images)} images")
images = images[1:]

Extracted 11 images


In [41]:
image_table_pattern = re.compile(
    r"\|\s*\|\s*\n\|[:\-]+\|\n\|(.*?)\|\n?", flags=re.DOTALL
)

md_text = Path(input_MD).read_text(encoding="utf-8")
captions = [m.group(1).strip() for m in image_table_pattern.finditer(md_text)]

print(f"Extracted {len(captions)} captions")



Extracted 10 captions


In [42]:
def describe_image(image_path, caption):
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {
                        "type": "text",
                        "text": f"""
                                You are helping annotate a Machine Learning Wikipedia article.
                                The image has the following caption: "{caption}"
                                Describe the image only in a way consistent with the caption and the topic in a single paragraph. 
                                """
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content


descriptions = []

for i in range(len(images)):
    desc = describe_image(images[i]["file"], captions[i])
    descriptions.append({
        "caption": captions[i],
        "description": desc
    })

print(f"Generated {len(descriptions)} image descriptions")


Generated 10 image descriptions


In [43]:
appendix = "\n\n---\n\n## Visual Context Data\n\n"

print("Generated Descriptions:\n")

for i, item in enumerate(descriptions, start=1):
    print(f"[Image {i}]")
    print("Caption:", item["caption"])
    print("Description:", item["description"])

    appendix += f"\n**[Image {i} Caption]:** {item['caption']}\n"
    appendix += f"**Description:** {item['description']}\n"

final_text = md_text + appendix
Path(output_MD).write_text(final_text, encoding="utf-8")

print("\n Added descriptions to markdown")


Generated Descriptions:

[Image 1]
Caption: Deep learning is a subset of machine learning, which is itself a subset of artificial intelligence.[20]
Description: A simple diagram of three nested circles shows their hierarchical relationship: the largest outer circle, shaded pale yellow, is labeled "Artificial Intelligence"; inside it a medium green circle labeled "Machine Learning" sits concentric with the outer circle; and within that a smaller light-blue circle labeled "Deep Learning" occupies the center, visually conveying that deep learning is a subset of machine learning, which in turn is a subset of artificial intelligence.
[Image 2]
Caption: In supervised learning, the training data is labelled with the expected answers, while in unsupervised learning, the model identifies patterns or structures in unlabelled data.
Description: A split diagram contrasts supervised and unsupervised learning: the left panel shows two labeled classes of training examples (blue squares and red triang

In [48]:
t = Path(output_MD).read_text(encoding="utf-8")

# Removing entire "References" section
t = re.sub(r"(?is)(^|\n)#+\s*\**references\**.*", "", t)

# Removing timestamp header lines like: "12/5/25, 2:47 PM Machine learning - Wikipedia"
t = re.sub(r"^\s*\d{1,2}/\d{1,2}/\d{2,4}.*Wikipedia\s*$", "", t, flags=re.MULTILINE)

# Removing plain URLs
t = re.sub(r"https?://[^\s]+", "", t)

# Removing standalone page-number lines like "12/24"
t = re.sub(r"^\s*\d+/\d+\s*$", "", t, flags=re.MULTILINE)

# Removing citation markers like [1], [2, 3], [4,5,6]
t = re.sub(r"\[\d+(,\s*\ad+)*\]", "", t)

# Removing numeric ranges like ": 5–10" or ": 12, 14"
t = re.sub(r": \d+(–\d+)?(, \d+)*", "", t)

# Removing empty markdown tables generated by converters
t = re.sub(r"\|\s*\|\s*\n\|[:\-]+\|\n\|(.*?)\|\n?", "", t, flags=re.DOTALL)

# Removing inline image markdown: ![alt](path)
t = re.sub(r"!\[.*?\]\(.*?\)", " ", t, flags=re.DOTALL)

# Fixing broken words split across newlines:
t = re.sub(r"([a-zA-Z,])\n\s*([a-z])", r"\1 \2", t)

# Collapse excessive blank lines
t = re.sub(r"\n\s*\n\s*\n+", "\n\n", t)
t = t.strip()


In [49]:
final_text = t + appendix
Path(output_MD).write_text(final_text, encoding="utf-8")

print("cleaned markdown saved as"+output_MD )



cleaned markdown saved as../output_cleaned_final.md
